# Etapa 1 — Exploración del dataset

Carga del PhiUSIIL Phishing URL Dataset y chequeos básicos. Solo vamos a usar la columna `URL`.

In [1]:
import pandas as pd
import tldextract

RUTA_CSV = "PhiUSIIL_Phishing_URL_Dataset.csv"

# El CSV viene con BOM UTF-8; con "utf-8-sig" la primera columna queda como "FILENAME"
df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")

# Chequeo: label debería tener solo los valores 0 y 1
valores_label = set(df["label"].unique())
if valores_label != {0, 1}:
    print(f"ATENCIÓN: 'label' tiene valores inesperados: {valores_label}")

# En el dataset original label = 1 es LEGÍTIMA y label = 0 es PHISHING.
# Invertimos para trabajar siempre con phishing = 1.
df["phishing"] = 1 - df["label"]

In [2]:
# Forma del dataset
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

Filas: 235,795
Columnas: 57


In [3]:
# Columnas disponibles (solo usaremos URL; el resto son features precalculadas que ignoramos)
print(list(df.columns))

['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label', 'phishing']


In [4]:
# Balance de clases
nombres_clase = {0: "legítima", 1: "phishing"}
balance = pd.DataFrame({
    "cantidad": df["phishing"].value_counts(),
    "proporción": df["phishing"].value_counts(normalize=True).round(4),
}).rename(index=nombres_clase)
balance

,cantidad,proporción
phishing,,
legítima,134850,0.5719
phishing,100945,0.4281


In [5]:
# 5 URLs de ejemplo de cada clase
for clase, nombre in nombres_clase.items():
    print(f"--- {nombre} ---")
    ejemplos = df.loc[df["phishing"] == clase, "URL"].sample(5, random_state=42)
    for url in ejemplos:
        print(url)
    print()

--- legítima ---
https://www.atelierozmoz.be
https://www.diemon.com
https://www.wausauschools.org
https://www.paademode.com
https://www.boxturtles.com

--- phishing ---
http://www.soeme.com
http://www.v2cde1b0d66c767a23cfb34c14552836d3.ws
https://locateme.co.nz/wp-content/jam/ichiemagiksouthwest123.html
http://www.kwan078lj.web.app
https://jumptruegallery.000webhostapp.com/download-documents



In [6]:
# Dominios registrables únicos (misma definición que usa features.py)
from features import dominio_registrable

dominios = df["URL"].map(dominio_registrable)
print(f"Dominios registrables únicos (total): {dominios.nunique():,}")
for clase, nombre in nombres_clase.items():
    print(f"Dominios registrables únicos ({nombre}): {dominios[df['phishing'] == clase].nunique():,}")

Dominios registrables únicos (total): 175,509
Dominios registrables únicos (legítima): 132,117
Dominios registrables únicos (phishing): 43,512


# Etapa 2 — Prueba de extract_features

Probamos la función de `features.py` con algunas URLs a mano (todavía no con el dataset completo).

In [7]:
from features import extract_features

urls_prueba = [
    "https://www.bna.com.ar",
    "http://bna-homebanking-verificar.xyz/login",
    "https://www.afip.gob.ar/",                            # sufijo .gob.ar
    "http://192.168.0.1:8080/paypal/login.php?id=1&t=2",   # IP, puerto, marca y parámetros
    "https://bit.ly/3xYz12",                               # acortador
    "https://storage.googleapis.com/algo",                 # dominio oficial de Google
    "http://paypal-login-seguro.com",                      # marca fuera de su dominio
]

# Una columna por URL para compararlas lado a lado
pd.DataFrame([extract_features(u) for u in urls_prueba], index=urls_prueba).T

,https://www.bna.com.ar,http://bna-homebanking-verificar.xyz/login,https://www.afip.gob.ar/,http://192.168.0.1:8080/paypal/login.php?id=1&t=2,https://bit.ly/3xYz12,https://storage.googleapis.com/algo,http://paypal-login-seguro.com
longitud_url,22.000000,42.000000,24.000000,49.000000,21.000000,35.000000,30.000000
longitud_dominio,14.000000,29.000000,15.000000,11.000000,6.000000,22.000000,23.000000
cant_puntos,3.000000,1.000000,3.000000,4.000000,1.000000,2.000000,1.000000
cant_guiones,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,2.000000
cant_digitos,0.000000,0.000000,0.000000,14.000000,3.000000,0.000000,0.000000
cant_especiales,0.000000,0.000000,0.000000,4.000000,0.000000,0.000000,0.000000
proporcion_digitos,0.000000,0.000000,0.000000,0.285714,0.142857,0.000000,0.000000
cant_subdominios,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000
usa_ip,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
usa_https,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.000000


**Nota: posible artefacto del dataset.** En PhiUSIIL muchas URLs legítimas son solo la página de inicio (`https://www.dominio.com`) y el phishing suele tener path. Por eso `longitud_path`, `profundidad_path`, `usa_https` y `longitud_url` pueden separar las clases por cómo se armó el dataset y no por una señal real de phishing. Si más adelante las métricas dan casi perfectas, es la primera causa a revisar.

# Etapa 3 — Features del dataset completo

Aplicamos `extract_features` a todas las URLs y guardamos el resultado en `data/features.parquet`.

In [8]:
import time

import numpy as np

from features import dominio_registrable, extract_features

inicio = time.perf_counter()
X = pd.DataFrame([extract_features(u) for u in df["URL"]])
y = df["phishing"].reset_index(drop=True)
dominio = df["URL"].map(dominio_registrable).reset_index(drop=True)
duracion = time.perf_counter() - inicio

print(f"Forma de X: {X.shape}")
print(f"Tiempo: {duracion:.1f} s ({len(X) / duracion:,.0f} URLs/s)")

Forma de X: (235795, 23)
Tiempo: 14.4 s (16,320 URLs/s)


In [9]:
# Controles de calidad (solo se muestran; no se elimina nada)
nulos = X.isna().sum()
print(f"Valores nulos: {nulos.sum()}")
if nulos.sum():
    print(nulos[nulos > 0])

infinitos = pd.Series(np.isinf(X.to_numpy(dtype=float)).sum(axis=0), index=X.columns)
print(f"Valores infinitos: {infinitos.sum()}")
if infinitos.sum():
    print(infinitos[infinitos > 0])

print(f"Filas con dominio vacío: {(dominio == '').sum()}")

urls = df["URL"].reset_index(drop=True)
print(f"Filas con URL duplicada: {urls.duplicated().sum():,}")
print(f"URLs distintas que se repiten: {urls[urls.duplicated(keep=False)].nunique():,}")
etiquetas_por_url = df.groupby("URL")["phishing"].nunique()
print(f"URLs duplicadas con etiquetas distintas: {(etiquetas_por_url > 1).sum():,}")

Valores nulos: 0
Valores infinitos: 0
Filas con dominio vacío: 0
Filas con URL duplicada: 425
URLs distintas que se repiten: 425


URLs duplicadas con etiquetas distintas: 0


**Columnas que NO son features** en `data/features.parquet`:
- `URL`: se guarda solo para inspeccionar los errores del modelo en la etapa 4.
- `phishing`: el target (1 = phishing).
- `dominio`: dominio registrable, se usa como grupo en `GroupShuffleSplit`.

Al entrenar, hay que excluir estas tres columnas de `X`.

In [10]:
from pathlib import Path

RUTA_FEATURES = Path("data") / "features.parquet"
RUTA_FEATURES.parent.mkdir(exist_ok=True)

df_features = X.assign(URL=urls, phishing=y, dominio=dominio)
df_features.to_parquet(RUTA_FEATURES, index=False)

print(f"Guardado en {RUTA_FEATURES}")
print(f"Forma: {df_features.shape}")
print(f"Tamaño: {RUTA_FEATURES.stat().st_size / 1e6:.1f} MB")

Guardado en data\features.parquet
Forma: (235795, 26)
Tamaño: 10.4 MB


# Etapa 4 — Entrenamiento y evaluación

Cargamos las features guardadas, dividimos por dominio registrable, comparamos dos modelos y medimos cuánto dependen del atajo del dataset. El modelo todavía no se guarda.

In [11]:
import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit

from features import extract_features

datos = pd.read_parquet("data/features.parquet")

# URL, phishing y dominio no son features
NO_FEATURES = ["URL", "phishing", "dominio"]
X = datos.drop(columns=NO_FEATURES)
y = datos["phishing"]
grupos = datos["dominio"]

assert X.shape[1] == 23, f"Se esperaban 23 features y hay {X.shape[1]}"
print(f"Forma de X: {X.shape}")

Forma de X: (235795, 23)


In [12]:
# Dominios antes de dividir
por_dominio = pd.crosstab(grupos, y).rename(columns={0: "legítima", 1: "phishing"})
por_dominio["total"] = por_dominio["legítima"] + por_dominio["phishing"]
por_dominio = por_dominio.sort_values("total", ascending=False)

print("15 dominios con más URLs:")
display(por_dominio.head(15))

ambas_clases = por_dominio[(por_dominio["legítima"] > 0) & (por_dominio["phishing"] > 0)]
print(f"Dominios con URLs de ambas clases: {len(ambas_clases)}")
display(ambas_clases.head(20))

15 dominios con más URLs:


phishing,legítima,phishing,total
dominio,,,
web.app,0,5754,5754
firebaseapp.com,0,5594,5594
repl.co,0,3754,3754
weeblysite.com,0,3097,3097
ipfs.io,0,1559,1559
workers.dev,0,1438,1438
square.site,0,1199,1199
dweb.link,0,975,975
xsph.ru,0,928,928


Dominios con URLs de ambas clases: 120


phishing,legítima,phishing,total
dominio,,,
duckdns.org,1,642,643
pinata.cloud,1,275,276
cutt.ly,1,35,36
eu.org,3,25,28
uk.com,22,1,23
wikidot.com,1,18,19
org.ru,3,13,16
washington.edu,11,1,12
clickfunnels.com,1,9,10


In [13]:
# División por dominio registrable (el 20% es de dominios, no de filas)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(X, y, groups=grupos))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
dom_train, dom_test = grupos.iloc[idx_train], grupos.iloc[idx_test]

compartidos = set(dom_train) & set(dom_test)
print(f"Dominios compartidos entre train y test: {len(compartidos)}")
assert not compartidos, "Hay dominios en ambos conjuntos"

print(f"Filas train: {len(X_train):,} ({len(X_train) / len(X):.1%})")
print(f"Filas test:  {len(X_test):,} ({len(X_test) / len(X):.1%})")

nombres_clase = {0: "legítima", 1: "phishing"}
balance_split = pd.DataFrame({
    "train": y_train.value_counts(),
    "train %": y_train.value_counts(normalize=True).round(4),
    "test": y_test.value_counts(),
    "test %": y_test.value_counts(normalize=True).round(4),
}).rename(index=nombres_clase)
display(balance_split)

Dominios compartidos entre train y test: 0
Filas train: 193,843 (82.2%)
Filas test:  41,952 (17.8%)


,train,train %,test,test %
phishing,,,,
legítima,107904,0.5567,26946,0.6423
phishing,85939,0.4433,15006,0.3577


In [14]:
def evaluar(nombre, modelo, X_te, y_te):
    """Imprime reporte, matriz de confusión, ROC-AUC y FPR; devuelve las métricas."""
    proba = modelo.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    print(f"===== {nombre} =====")
    print(classification_report(y_te, pred, target_names=["legítima", "phishing"], digits=4))

    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    matriz = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["real legítima", "real phishing"],
        columns=["pred. legítima", "pred. phishing"],
    )
    print(matriz, "\n")

    metricas = {
        "modelo": nombre,
        "roc_auc": roc_auc_score(y_te, proba),
        "recall_phishing": recall_score(y_te, pred),
        "precision_phishing": precision_score(y_te, pred),
        "fpr": fp / (fp + tn),
        "accuracy": accuracy_score(y_te, pred),
        "proba": proba,
    }
    print(f"ROC-AUC: {metricas['roc_auc']:.4f} | tasa de falsos positivos: {metricas['fpr']:.4f}\n")
    return metricas


def tabla(resultados):
    """Tabla comparativa sin la columna de probabilidades."""
    return pd.DataFrame([{k: v for k, v in r.items() if k != "proba"} for r in resultados]).set_index("modelo").round(4)

In [15]:
# Entrenamiento de los dos modelos con las 23 features
modelos = {
    "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

resultados = []
entrenados = {}
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    print(f"{nombre}: entrenado en {time.perf_counter() - inicio:.1f} s")
    entrenados[nombre] = modelo
    resultados.append(evaluar(nombre, modelo, X_test, y_test))

RandomForest: entrenado en 15.5 s


===== RandomForest =====
              precision    recall  f1-score   support

    legítima     0.9949    0.9986    0.9967     26946
    phishing     0.9974    0.9908    0.9941     15006

    accuracy                         0.9958     41952
   macro avg     0.9961    0.9947    0.9954     41952
weighted avg     0.9958    0.9958    0.9958     41952

               pred. legítima  pred. phishing
real legítima           26907              39
real phishing             138           14868 

ROC-AUC: 0.9968 | tasa de falsos positivos: 0.0014



HistGradientBoosting: entrenado en 1.3 s
===== HistGradientBoosting =====
              precision    recall  f1-score   support

    legítima     0.9946    0.9990    0.9968     26946
    phishing     0.9983    0.9903    0.9943     15006

    accuracy                         0.9959     41952
   macro avg     0.9964    0.9947    0.9956     41952
weighted avg     0.9959    0.9959    0.9959     41952

               pred. legítima  pred. phishing
real legítima           26920              26
real phishing             145           14861 

ROC-AUC: 0.9979 | tasa de falsos positivos: 0.0010



In [16]:
# Comparación y elección del mejor (ROC-AUC; si empatan a 4 decimales, menor FPR)
display(tabla(resultados))

mejor_res = min(resultados, key=lambda r: (-round(r["roc_auc"], 4), r["fpr"]))
nombre_mejor = mejor_res["modelo"]
mejor = entrenados[nombre_mejor]
print(f"Mejor modelo: {nombre_mejor}")

sospechosos = [r["modelo"] for r in resultados if r["roc_auc"] >= 0.99 or r["accuracy"] >= 0.99]
if sospechosos:
    print(
        f"\nATENCIÓN: métricas sospechosamente altas en {', '.join(sospechosos)}.\n"
        "Causas probables: atajo https/www/path (las legítimas son casi siempre la home con HTTPS),\n"
        "hostings gratuitos que concentran phishing y la forma en que se armó el dataset."
    )

,roc_auc,recall_phishing,precision_phishing,fpr,accuracy
modelo,,,,,
RandomForest,0.9968,0.9908,0.9974,0.0014,0.9958
HistGradientBoosting,0.9979,0.9903,0.9983,0.0010,0.9959


Mejor modelo: HistGradientBoosting

ATENCIÓN: métricas sospechosamente altas en RandomForest, HistGradientBoosting.
Causas probables: atajo https/www/path (las legítimas son casi siempre la home con HTTPS),
hostings gratuitos que concentran phishing y la forma en que se armó el dataset.


In [17]:
# Prueba del atajo del dataset
ATAJO = ["longitud_path", "profundidad_path", "usa_https", "longitud_url"]
TRIVIAL = ["usa_https", "profundidad_path"]

sin_atajo = clone(mejor).fit(X_train.drop(columns=ATAJO), y_train)
res_sin_atajo = evaluar(f"{nombre_mejor} sin atajo", sin_atajo, X_test.drop(columns=ATAJO), y_test)

# Modelo trivial de diagnóstico: solo HTTPS y profundidad del path
trivial = clone(mejor).fit(X_train[TRIVIAL], y_train)
res_trivial = evaluar(f"{nombre_mejor} trivial (solo {' + '.join(TRIVIAL)})", trivial, X_test[TRIVIAL], y_test)

comparacion_atajo = tabla([mejor_res, res_sin_atajo, res_trivial])
display(comparacion_atajo)
print("Diferencia sin atajo - completo:")
display((comparacion_atajo.iloc[1] - comparacion_atajo.iloc[0]).round(4).to_frame("diferencia"))

===== HistGradientBoosting sin atajo =====
              precision    recall  f1-score   support

    legítima     0.8453    0.9635    0.9006     26946
    phishing     0.9125    0.6835    0.7816     15006

    accuracy                         0.8633     41952
   macro avg     0.8789    0.8235    0.8411     41952
weighted avg     0.8694    0.8633    0.8580     41952

               pred. legítima  pred. phishing
real legítima           25963             983
real phishing            4750           10256 

ROC-AUC: 0.8651 | tasa de falsos positivos: 0.0365



===== HistGradientBoosting trivial (solo usa_https + profundidad_path) =====
              precision    recall  f1-score   support

    legítima     0.9331    1.0000    0.9654     26946
    phishing     1.0000    0.8713    0.9312     15006

    accuracy                         0.9539     41952
   macro avg     0.9665    0.9356    0.9483     41952
weighted avg     0.9570    0.9539    0.9532     41952

               pred. legítima  pred. phishing
real legítima           26946               0
real phishing            1932           13074 

ROC-AUC: 0.9356 | tasa de falsos positivos: 0.0000



,roc_auc,recall_phishing,precision_phishing,fpr,accuracy
modelo,,,,,
HistGradientBoosting,0.9979,0.9903,0.9983,0.0010,0.9959
HistGradientBoosting sin atajo,0.8651,0.6835,0.9125,0.0365,0.8633
HistGradientBoosting trivial (solo usa_https + profundidad_path),0.9356,0.8713,1.0000,0.0000,0.9539


Diferencia sin atajo - completo:


,diferencia
roc_auc,-0.1328
recall_phishing,-0.3068
precision_phishing,-0.0858
fpr,0.0355
accuracy,-0.1326


In [18]:
# Punto de operación con FPR <= 1% (en una extensión los falsos positivos son muy costosos)
def recall_con_fpr_max(y_te, proba, fpr_max=0.01):
    """Mayor recall alcanzable con FPR <= fpr_max; devuelve (recall, umbral, fpr)."""
    fpr, tpr, umbrales = roc_curve(y_te, proba)
    validos = np.where(fpr <= fpr_max)[0]
    i = validos[np.argmax(tpr[validos])]
    return tpr[i], umbrales[i], fpr[i]


umbrales_op = {}
filas = []
for res in [mejor_res, res_sin_atajo]:
    recall, umbral, fpr_real = recall_con_fpr_max(y_test, res["proba"])
    umbrales_op[res["modelo"]] = umbral
    filas.append({"modelo": res["modelo"], "recall_phishing": recall, "umbral": umbral, "fpr_real": fpr_real})

print("Recall de phishing con FPR <= 1%:")
display(pd.DataFrame(filas).set_index("modelo").round(4))

Recall de phishing con FPR <= 1%:


,recall_phishing,umbral,fpr_real
modelo,,,
HistGradientBoosting,0.9923,0.0790,0.0092
HistGradientBoosting sin atajo,0.5962,0.8017,0.0100


In [19]:
# URLs de prueba: las 7 de la etapa 2 + 5 legítimas con path (navegación real)
urls_prueba = {
    "https://www.bna.com.ar": "legítima",
    "http://bna-homebanking-verificar.xyz/login": "phishing",
    "https://www.afip.gob.ar/": "legítima",
    "http://192.168.0.1:8080/paypal/login.php?id=1&t=2": "phishing",
    "https://bit.ly/3xYz12": "acortador",
    "https://storage.googleapis.com/algo": "legítima",
    "http://paypal-login-seguro.com": "phishing",
    "https://www.mercadolibre.com.ar/ofertas": "legítima",
    "https://www.argentina.gob.ar/anses/jubilaciones": "legítima",
    "https://github.com/scikit-learn/scikit-learn/issues": "legítima",
    "https://www.bna.com.ar/Personas/Prestamos": "legítima",
    "https://es.wikipedia.org/wiki/Phishing": "legítima",
}

X_urls = pd.DataFrame([extract_features(u) for u in urls_prueba])[X.columns]
versiones = {
    "completo": (mejor, X_urls, umbrales_op[mejor_res["modelo"]]),
    "sin atajo": (sin_atajo, X_urls.drop(columns=ATAJO), umbrales_op[res_sin_atajo["modelo"]]),
}

prueba = pd.DataFrame({"esperado": list(urls_prueba.values())}, index=list(urls_prueba))
es_legitima = prueba["esperado"] == "legítima"
for nombre, (modelo, X_v, umbral) in versiones.items():
    p = modelo.predict_proba(X_v)[:, 1]
    prueba[f"p_phishing ({nombre})"] = p.round(3)
    # Marca falsos positivos en URLs legítimas: con umbral 0.5 y con el umbral de FPR <= 1%
    marcas = []
    for legit, prob in zip(es_legitima, p):
        m = [etiqueta for etiqueta, u in [("0.5", 0.5), ("FPR≤1%", umbral)] if legit and prob >= u]
        marcas.append(f"FP ({', '.join(m)})" if m else "")
    prueba[f"FP ({nombre})"] = marcas
    print(f"Umbral FPR ≤ 1% ({nombre}): {umbral:.3f}")

display(prueba)

Umbral FPR ≤ 1% (completo): 0.079
Umbral FPR ≤ 1% (sin atajo): 0.802


,esperado,p_phishing (completo),FP (completo),p_phishing (sin atajo),FP (sin atajo)
https://www.bna.com.ar,legítima,0.0,,0.058,
http://bna-homebanking-verificar.xyz/login,phishing,1.0,,1.000,
https://www.afip.gob.ar/,legítima,1.0,"FP (0.5, FPR≤1%)",0.097,
http://192.168.0.1:8080/paypal/login.php?id=1&t=2,phishing,1.0,,1.000,
https://bit.ly/3xYz12,acortador,1.0,,1.000,
https://storage.googleapis.com/algo,legítima,1.0,"FP (0.5, FPR≤1%)",0.146,
http://paypal-login-seguro.com,phishing,1.0,,1.000,
https://www.mercadolibre.com.ar/ofertas,legítima,1.0,"FP (0.5, FPR≤1%)",0.187,
https://www.argentina.gob.ar/anses/jubilaciones,legítima,1.0,"FP (0.5, FPR≤1%)",0.146,
https://github.com/scikit-learn/scikit-learn/issues,legítima,1.0,"FP (0.5, FPR≤1%)",1.000,"FP (0.5, FPR≤1%)"


## Conclusiones de la etapa 4

**Las métricas del modelo completo son sospechosamente perfectas y no representan el uso real.** HistGradientBoosting (el mejor) da ROC-AUC 0.998, recall de phishing 0.990 y FPR 0.10% en test. Pero:

- **El modelo trivial con solo `usa_https` y `profundidad_path` ya logra ROC-AUC 0.936 con FPR 0%.** Ninguna URL legítima del test tiene path o usa HTTP. El dataset está armado así: las legítimas son homes `https://www.dominio.tld`.
- **En las URLs de prueba, el modelo completo marca como phishing (p = 1.0) a TODAS las legítimas con path**, e incluso a `https://www.afip.gob.ar/` solo por la `/` final. Son falsos positivos: afip.gob.ar/, storage.googleapis.com, mercadolibre.com.ar/ofertas, argentina.gob.ar/anses/jubilaciones, github.com/…/issues, bna.com.ar/Personas/Prestamos y es.wikipedia.org/wiki/Phishing. En una extensión, eso significa alertar en casi cualquier página que visite el usuario.

**Versión sin ATAJO** (sin `longitud_path`, `profundidad_path`, `usa_https` ni `longitud_url`):
- En test baja a ROC-AUC 0.865, recall 0.684 y FPR 3.65% con umbral 0.5. Con **FPR ≤ 1%**, el recall es **0.596** con umbral **0.80**.
- En las URLs de prueba se comporta de forma razonable: los 3 phishing y el acortador dan ~1.0, y las legítimas quedan entre 0.06 y 0.19. **Única excepción: `https://github.com/scikit-learn/scikit-learn/issues` da 1.0, un falso positivo** incluso con el umbral de 0.80.
- No elimina el atajo del todo: `cant_puntos`, `cant_digitos`, `cant_especiales`, `proporcion_digitos` y `cant_parametros` también se calculan sobre la URL completa, así que el path sigue influyendo de forma indirecta. Las métricas de test de esta versión probablemente siguen siendo optimistas.

**Recomendación:** usar la **versión sin ATAJO con umbral ~0.80** (punto de operación con FPR ≤ 1%). Tiene menos recall en el papel, pero el modelo completo es inutilizable en navegación real porque confunde "tiene path" con "es phishing". El recall que falta lo pueden compensar las reglas y el LLM del backend. Queda pendiente para una etapa futura: calcular esos conteos solo sobre el hostname y revisar el falso positivo de github.

Nota: con la división por dominio, el test quedó con 64% de legítimas contra 56% en train, porque hostings grandes que son 100% phishing (web.app, firebaseapp.com, repl.co…) caen enteros de un lado.

# Etapa 4b — Modelo basado solo en el dominio

Como ninguna URL legítima del dataset tiene path, cualquier feature calculada sobre la URL completa es un atajo. Acá usamos la versión 4b (congelada como `extract_features_dominio_4b`): mira solo el hostname (sin esquema, path, query ni puerto) y además saca el prefijo `www.`.

In [20]:
# Versión 4b congelada (solo para comparar; la versión vigente es la de la etapa 4c)
from features import extract_features_dominio_4b

# Prueba con las 12 URLs
pd.DataFrame([extract_features_dominio_4b(u) for u in urls_prueba], index=list(urls_prueba)).T

,https://www.bna.com.ar,http://bna-homebanking-verificar.xyz/login,https://www.afip.gob.ar/,http://192.168.0.1:8080/paypal/login.php?id=1&t=2,https://bit.ly/3xYz12,https://storage.googleapis.com/algo,http://paypal-login-seguro.com,https://www.mercadolibre.com.ar/ofertas,https://www.argentina.gob.ar/anses/jubilaciones,https://github.com/scikit-learn/scikit-learn/issues,https://www.bna.com.ar/Personas/Prestamos,https://es.wikipedia.org/wiki/Phishing
longitud_dominio,10.000000,29.000000,11.000000,11.000000,6.000000,22.000000,23.000000,19.000000,16.00000,10.000000,10.000000,16.00000
cant_puntos,2.000000,1.000000,2.000000,3.000000,1.000000,2.000000,1.000000,2.000000,2.00000,1.000000,2.000000,2.00000
cant_guiones,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.00000,0.000000,0.000000,0.00000
cant_digitos,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000
proporcion_digitos,0.000000,0.000000,0.000000,0.727273,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000
cant_subdominios,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.00000
usa_ip,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000
tiene_punycode,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000
entropia_dominio,2.921928,4.090234,3.095795,2.594907,2.584963,3.516028,3.882045,3.366091,3.20282,3.321928,2.921928,3.45282
tld_ar,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,0.000000,1.000000,0.00000


In [21]:
# Recalcular las features del dataset con la versión de dominio 4b
from pathlib import Path

base = pd.read_parquet("data/features.parquet", columns=["URL", "phishing", "dominio"])

inicio = time.perf_counter()
X_dom = pd.DataFrame([extract_features_dominio_4b(u) for u in base["URL"]])
duracion = time.perf_counter() - inicio
print(f"Forma de X_dom: {X_dom.shape}")
print(f"Tiempo: {duracion:.1f} s")
print(f"Valores nulos: {X_dom.isna().sum().sum()}")
print(f"Valores infinitos: {np.isinf(X_dom.to_numpy(dtype=float)).sum()}")

RUTA_FEATURES_DOMINIO_4B = Path("data") / "features_dominio_4b.parquet"
X_dom.assign(URL=base["URL"], phishing=base["phishing"], dominio=base["dominio"]).to_parquet(
    RUTA_FEATURES_DOMINIO_4B, index=False
)
print(f"Guardado en {RUTA_FEATURES_DOMINIO_4B} ({RUTA_FEATURES_DOMINIO_4B.stat().st_size / 1e6:.1f} MB)")

Forma de X_dom: (235795, 15)
Tiempo: 10.5 s
Valores nulos: 0
Valores infinitos: 0
Guardado en data\features_dominio_4b.parquet (9.7 MB)


In [22]:
# Diagnóstico del atajo "www" sobre el hostname ORIGINAL (antes de normalizar)
tiene_www = base["URL"].str.lower().str.match(r"^(?:[a-z][a-z0-9+.-]*://)?www\.")
print("Proporción de hostnames que empiezan con 'www.':")
display(tiene_www.groupby(base["phishing"]).mean().rename(index=nombres_clase).round(4).to_frame("con www."))

Proporción de hostnames que empiezan con 'www.':


,con www.
phishing,
legítima,1.0000
phishing,0.4136


In [23]:
# Mismo split por dominio que la etapa 4
assert (base["dominio"].to_numpy() == grupos.to_numpy()).all(), "El orden de filas no coincide con la etapa 4"
assert not set(base["dominio"].iloc[idx_train]) & set(base["dominio"].iloc[idx_test])

Xd_train, Xd_test = X_dom.iloc[idx_train], X_dom.iloc[idx_test]

modelo_dom = HistGradientBoostingClassifier(random_state=42).fit(Xd_train, y_train)
res_dom = evaluar("HGB dominio 4b", modelo_dom, Xd_test, y_test)

filas = []
for res in [res_sin_atajo, res_dom]:
    recall_op, umbral, fpr_real = recall_con_fpr_max(y_test, res["proba"])
    umbrales_op[res["modelo"]] = umbral
    filas.append({**{k: v for k, v in res.items() if k != "proba"},
                  "recall_fpr1%": recall_op, "umbral_fpr1%": umbral})

print("Comparación con la versión sin atajo de la etapa 4:")
display(pd.DataFrame(filas).set_index("modelo").round(4))

===== HGB dominio 4b =====
              precision    recall  f1-score   support

    legítima     0.8021    0.9653    0.8762     26946
    phishing     0.9019    0.5723    0.7003     15006

    accuracy                         0.8248     41952
   macro avg     0.8520    0.7688    0.7882     41952
weighted avg     0.8378    0.8248    0.8133     41952

               pred. legítima  pred. phishing
real legítima           26012             934
real phishing            6418            8588 

ROC-AUC: 0.8196 | tasa de falsos positivos: 0.0347

Comparación con la versión sin atajo de la etapa 4:


,roc_auc,recall_phishing,precision_phishing,fpr,accuracy,recall_fpr1%,umbral_fpr1%
modelo,,,,,,,
HistGradientBoosting sin atajo,0.8651,0.6835,0.9125,0.0365,0.8633,0.5962,0.8017
HGB dominio 4b,0.8196,0.5723,0.9019,0.0347,0.8248,0.4849,0.8445


In [24]:
def comparar_urls(urls, versiones):
    """p_phishing de cada versión para las URLs de prueba, marcando falsos positivos.

    versiones: {nombre: (modelo, X de las URLs, umbral con FPR <= 1%)}.
    Una legítima se marca como FP si supera 0.5 y/o el umbral de FPR <= 1%.
    """
    tabla_urls = pd.DataFrame({"esperado": list(urls.values())}, index=list(urls))
    es_legitima = tabla_urls["esperado"] == "legítima"
    for nombre, (modelo, X_v, umbral) in versiones.items():
        p = modelo.predict_proba(X_v)[:, 1]
        tabla_urls[f"p_phishing ({nombre})"] = p.round(3)
        marcas = []
        for legit, prob in zip(es_legitima, p):
            m = [etiqueta for etiqueta, u in [("0.5", 0.5), ("FPR≤1%", umbral)] if legit and prob >= u]
            marcas.append(f"FP ({', '.join(m)})" if m else "")
        tabla_urls[f"FP ({nombre})"] = marcas
        print(f"Umbral FPR ≤ 1% ({nombre}): {umbral:.3f}")
    return tabla_urls


# 12 URLs: modelo de dominio 4b vs. versión sin atajo de la etapa 4
X_urls_dom = pd.DataFrame([extract_features_dominio_4b(u) for u in urls_prueba])[X_dom.columns]
prueba_4b = comparar_urls(urls_prueba, {
    "sin atajo": (sin_atajo, X_urls.drop(columns=ATAJO), umbrales_op[res_sin_atajo["modelo"]]),
    "dominio": (modelo_dom, X_urls_dom, umbrales_op[res_dom["modelo"]]),
})
display(prueba_4b)

Umbral FPR ≤ 1% (sin atajo): 0.802
Umbral FPR ≤ 1% (dominio): 0.844


,esperado,p_phishing (sin atajo),FP (sin atajo),p_phishing (dominio),FP (dominio)
https://www.bna.com.ar,legítima,0.058,,0.097,
http://bna-homebanking-verificar.xyz/login,phishing,1.000,,0.989,
https://www.afip.gob.ar/,legítima,0.097,,0.141,
http://192.168.0.1:8080/paypal/login.php?id=1&t=2,phishing,1.000,,0.999,
https://bit.ly/3xYz12,acortador,1.000,,0.994,
https://storage.googleapis.com/algo,legítima,0.146,,0.880,"FP (0.5, FPR≤1%)"
http://paypal-login-seguro.com,phishing,1.000,,0.981,
https://www.mercadolibre.com.ar/ofertas,legítima,0.187,,0.340,
https://www.argentina.gob.ar/anses/jubilaciones,legítima,0.146,,0.238,
https://github.com/scikit-learn/scikit-learn/issues,legítima,1.000,"FP (0.5, FPR≤1%)",0.203,


In [25]:
def explicar_fp(tabla_urls, columna_fp, modelo, X_urls_v, X_tr, y_tr):
    """Para cada legítima marcada en columna_fp, reemplaza cada feature (una por vez)
    por la mediana de las legítimas de train y muestra cuánto baja p_phishing."""
    mediana_legit = X_tr[y_tr.to_numpy() == 0].median()
    marcadas = tabla_urls.index[tabla_urls[columna_fp] != ""]
    if len(marcadas) == 0:
        print(f"Ninguna URL legítima de prueba da alta ({columna_fp}).")
    for url in marcadas:
        fila = X_urls_v.iloc[[list(tabla_urls.index).index(url)]]
        p_base = modelo.predict_proba(fila)[0, 1]
        impactos = []
        for col in X_urls_v.columns:
            modificada = fila.copy()
            modificada[col] = mediana_legit[col]
            impactos.append({
                "feature": col,
                "valor_url": fila[col].iloc[0],
                "mediana_legítimas": mediana_legit[col],
                "baja_p": p_base - modelo.predict_proba(modificada)[0, 1],
            })
        print(f"\n{url}  (p_phishing = {p_base:.3f})")
        display(pd.DataFrame(impactos).sort_values("baja_p", ascending=False).head(5).round(3).set_index("feature"))


# ¿Qué feature causa cada legítima marcada como FP por el modelo de dominio 4b?
explicar_fp(prueba_4b, "FP (dominio)", modelo_dom, X_urls_dom, Xd_train, y_train)


https://storage.googleapis.com/algo  (p_phishing = 0.880)

,valor_url,mediana_legítimas,baja_p
feature,,,
cant_subdominios,1.000,0.000,0.708
longitud_dominio,22.000,15.000,0.271
entropia_dominio,3.516,3.345,0.036
cant_puntos,2.000,1.000,0.011
cant_guiones,0.000,0.000,0.000



https://es.wikipedia.org/wiki/Phishing  (p_phishing = 0.631)


,valor_url,mediana_legítimas,baja_p
feature,,,
cant_subdominios,1.000,0.000,0.467
longitud_dominio,16.000,15.000,0.026
entropia_dominio,3.453,3.345,0.005
cant_guiones,0.000,0.000,0.000
cant_digitos,0.000,0.000,0.000


## Conclusiones de la etapa 4b

**Atajo de `www` confirmado:** el 100% de las legítimas del dataset empiezan con `www.`, contra el 41% del phishing. Por eso normalizarlo era necesario.

**En test, el modelo de dominio rinde menos que la versión sin atajo**, pero esa versión sigue viendo el path de forma indirecta y el de dominio no:

| | ROC-AUC | recall (0.5) | FPR (0.5) | recall con FPR ≤ 1% | umbral |
|---|---|---|---|---|---|
| sin atajo (etapa 4) | 0.865 | 0.684 | 3.65% | 0.596 | 0.80 |
| dominio (4b) | 0.820 | 0.572 | 3.47% | 0.485 | 0.84 |

Las métricas ya no dan sospechosamente perfectas. Es un resultado más honesto de lo que se puede saber mirando solo el hostname.

**En las 12 URLs:**
- Detecta los 3 phishing y el acortador (p ≥ 0.98). Todas las legítimas de dominio "plano" quedan bajas: bna, afip, mercadolibre, argentina.gob.ar y github (0.10–0.34).
- **Se corrige el falso positivo de github** (1.0 → 0.20). Venía de las features de la URL completa.
- **Falsos positivos nuevos:**
  - `https://storage.googleapis.com/algo` da **0.88** y supera el umbral de 0.84;
  - `https://es.wikipedia.org/wiki/Phishing` da **0.63**; supera 0.5 pero no el umbral.

**Qué feature los causa:** `cant_subdominios`. Reemplazarla por la mediana de las legítimas (0) baja p en 0.71 (googleapis) y 0.47 (wikipedia); en googleapis también influye `longitud_dominio`, con −0.27. Es un **atajo residual del dataset**: después de sacar `www`, solo el 2.9% de las legítimas tiene algún subdominio, contra el 54% del phishing. Las legítimas de PhiUSIIL son homes `www.dominio.tld`, pero en la navegación real los subdominios legítimos son comunes (es.wikipedia.org, docs.google.com, storage.googleapis.com).

**Recomendación:** preferir el **modelo de dominio con umbral ~0.84**. Rinde menos en test, pero es el único que no penaliza el path, que es lo más frecuente al navegar. Su debilidad conocida son los subdominios legítimos. Hay dos opciones para una etapa siguiente: sacar o acotar `cant_subdominios` y medir el costo, o completar el dataset con URLs legítimas que tengan subdominios.

> **Actualización:** esta recomendación queda reemplazada por la de la etapa 4c (más abajo).

# Etapa 4c — Features estructurales sobre el dominio registrable

`extract_features_dominio` (versión vigente) calcula longitud, puntos, guiones, dígitos y entropía sobre el **dominio registrable**, y ya no tiene `cant_subdominios`. Las palabras sospechosas y la marca siguen mirando el **hostname completo**, porque en hostings gratuitos la marca suele ir en el subdominio (`paypal-login.web.app`).

In [26]:
from features import extract_features_dominio

urls_prueba_13 = {**urls_prueba, "http://paypal-login.web.app": "phishing"}

pd.DataFrame([extract_features_dominio(u) for u in urls_prueba_13], index=list(urls_prueba_13)).T

,https://www.bna.com.ar,http://bna-homebanking-verificar.xyz/login,https://www.afip.gob.ar/,http://192.168.0.1:8080/paypal/login.php?id=1&t=2,https://bit.ly/3xYz12,https://storage.googleapis.com/algo,http://paypal-login-seguro.com,https://www.mercadolibre.com.ar/ofertas,https://www.argentina.gob.ar/anses/jubilaciones,https://github.com/scikit-learn/scikit-learn/issues,https://www.bna.com.ar/Personas/Prestamos,https://es.wikipedia.org/wiki/Phishing,http://paypal-login.web.app
longitud_dominio,10.000000,29.000000,11.000000,11.000000,6.000000,14.000000,23.000000,19.000000,16.00000,10.000000,10.000000,13.000000,7.000000
cant_puntos,2.000000,1.000000,2.000000,3.000000,1.000000,1.000000,1.000000,2.000000,2.00000,1.000000,2.000000,1.000000,1.000000
cant_guiones,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
cant_digitos,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
proporcion_digitos,0.000000,0.000000,0.000000,0.727273,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
usa_ip,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
tiene_punycode,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
entropia_dominio,2.921928,4.090234,3.095795,2.594907,2.584963,3.324863,3.882045,3.366091,3.20282,3.321928,2.921928,3.334679,2.521641
tld_ar,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,0.000000,1.000000,0.000000,0.000000
tld_comun,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,0.000000


In [27]:
# Recalcular las features del dataset con la versión 4c
inicio = time.perf_counter()
X_4c = pd.DataFrame([extract_features_dominio(u) for u in base["URL"]])
duracion = time.perf_counter() - inicio
print(f"Forma de X_4c: {X_4c.shape}")
print(f"Tiempo: {duracion:.1f} s")
print(f"Valores nulos: {X_4c.isna().sum().sum()}")
print(f"Valores infinitos: {np.isinf(X_4c.to_numpy(dtype=float)).sum()}")

RUTA_FEATURES_DOMINIO = Path("data") / "features_dominio.parquet"
X_4c.assign(URL=base["URL"], phishing=base["phishing"], dominio=base["dominio"]).to_parquet(
    RUTA_FEATURES_DOMINIO, index=False
)
print(f"Guardado en {RUTA_FEATURES_DOMINIO} ({RUTA_FEATURES_DOMINIO.stat().st_size / 1e6:.1f} MB)")

Forma de X_4c: (235795, 14)
Tiempo: 10.4 s
Valores nulos: 0
Valores infinitos: 0
Guardado en data\features_dominio.parquet (9.3 MB)


In [28]:
# Mismo split que la etapa 4; dos modelos con las mismas features
assert not set(base["dominio"].iloc[idx_train]) & set(base["dominio"].iloc[idx_test])
X4c_train, X4c_test = X_4c.iloc[idx_train], X_4c.iloc[idx_test]

modelos_4c = {
    "HGB dominio 4c": HistGradientBoostingClassifier(random_state=42),
    "RF dominio 4c": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
}

res_4c, entrenados_4c, tiempo_entrenamiento = {}, {}, {}
for nombre, modelo in modelos_4c.items():
    inicio = time.perf_counter()
    modelo.fit(X4c_train, y_train)
    tiempo_entrenamiento[nombre] = time.perf_counter() - inicio
    print(f"{nombre}: entrenado en {tiempo_entrenamiento[nombre]:.1f} s")
    entrenados_4c[nombre] = modelo
    res_4c[nombre] = evaluar(nombre, modelo, X4c_test, y_test)
    umbrales_op[nombre] = recall_con_fpr_max(y_test, res_4c[nombre]["proba"])[1]

HGB dominio 4c: entrenado en 1.2 s
===== HGB dominio 4c =====
              precision    recall  f1-score   support

    legítima     0.7249    0.9050    0.8050     26946
    phishing     0.6920    0.3833    0.4934     15006

    accuracy                         0.7184     41952
   macro avg     0.7085    0.6442    0.6492     41952
weighted avg     0.7131    0.7184    0.6935     41952

               pred. legítima  pred. phishing
real legítima           24386            2560
real phishing            9254            5752 

ROC-AUC: 0.7154 | tasa de falsos positivos: 0.0950



RF dominio 4c: entrenado en 14.0 s


===== RF dominio 4c =====
              precision    recall  f1-score   support

    legítima     0.7248    0.8917    0.7996     26946
    phishing     0.6684    0.3920    0.4942     15006

    accuracy                         0.7130     41952
   macro avg     0.6966    0.6419    0.6469     41952
weighted avg     0.7046    0.7130    0.6904     41952

               pred. legítima  pred. phishing
real legítima           24027            2919
real phishing            9123            5883 

ROC-AUC: 0.7002 | tasa de falsos positivos: 0.1083



In [29]:
# Comparación pensando en producción (backend que responde a la extensión en tiempo real)
import io

import joblib

N_LLAMADAS = 200


def ms_por_url(modelo, X_una_fila):
    """Latencia media de predict_proba para UNA URL, como en una request real."""
    modelo.predict_proba(X_una_fila)  # calentamiento
    inicio = time.perf_counter()
    for _ in range(N_LLAMADAS):
        modelo.predict_proba(X_una_fila)
    return (time.perf_counter() - inicio) / N_LLAMADAS * 1000


def mb_serializado(modelo):
    """Tamaño del modelo serializado con joblib, solo en memoria (no se guarda en disco)."""
    buffer = io.BytesIO()
    joblib.dump(modelo, buffer)
    return len(buffer.getvalue()) / 1e6


candidatos = [
    ("HGB dominio 4b", modelo_dom, res_dom, Xd_test.iloc[[0]], float("nan")),
    *[(n, entrenados_4c[n], res_4c[n], X4c_test.iloc[[0]], tiempo_entrenamiento[n]) for n in modelos_4c],
]
filas = []
for nombre, modelo, res, una_fila, t_entrenamiento in candidatos:
    recall_op, umbral, _ = recall_con_fpr_max(y_test, res["proba"])
    filas.append({
        "modelo": nombre,
        "roc_auc": res["roc_auc"],
        "recall (0.5)": res["recall_phishing"],
        "fpr (0.5)": res["fpr"],
        "recall_fpr1%": recall_op,
        "umbral_fpr1%": umbral,
        "entrenamiento_s": t_entrenamiento,
        "ms_por_url": ms_por_url(modelo, una_fila),
        "tamaño_MB": mb_serializado(modelo),
    })
display(pd.DataFrame(filas).set_index("modelo").round(4))

# Costo de extraer las features de una URL (se suma a la latencia del modelo)
inicio = time.perf_counter()
for _ in range(N_LLAMADAS):
    extract_features_dominio("http://paypal-login.web.app")
print(f"extract_features_dominio: {(time.perf_counter() - inicio) / N_LLAMADAS * 1000:.3f} ms por URL")

,roc_auc,recall (0.5),fpr (0.5),recall_fpr1%,umbral_fpr1%,entrenamiento_s,ms_por_url,tamaño_MB
modelo,,,,,,,,
HGB dominio 4b,0.8196,0.5723,0.0347,0.4849,0.8445,NaN,1.9114,0.3695
HGB dominio 4c,0.7154,0.3833,0.0950,0.2246,0.9039,1.2305,1.8533,0.3696
RF dominio 4c,0.7002,0.3920,0.1083,0.2213,0.9249,13.9814,68.1565,222.0296


extract_features_dominio: 0.033 ms por URL


In [30]:
# 13 URLs: 4b vs. 4c (HGB y RF)
X_urls_4b = pd.DataFrame([extract_features_dominio_4b(u) for u in urls_prueba_13])[X_dom.columns]
X_urls_4c = pd.DataFrame([extract_features_dominio(u) for u in urls_prueba_13])[X_4c.columns]

prueba_4c = comparar_urls(urls_prueba_13, {
    "4b HGB": (modelo_dom, X_urls_4b, umbrales_op[res_dom["modelo"]]),
    **{n.replace("dominio ", ""): (entrenados_4c[n], X_urls_4c, umbrales_op[n]) for n in modelos_4c},
})
display(prueba_4c)

Umbral FPR ≤ 1% (4b HGB): 0.844
Umbral FPR ≤ 1% (HGB 4c): 0.904
Umbral FPR ≤ 1% (RF 4c): 0.925


,esperado,p_phishing (4b HGB),FP (4b HGB),p_phishing (HGB 4c),FP (HGB 4c),p_phishing (RF 4c),FP (RF 4c)
https://www.bna.com.ar,legítima,0.097,,0.092,,0.000,
http://bna-homebanking-verificar.xyz/login,phishing,0.989,,0.990,,0.970,
https://www.afip.gob.ar/,legítima,0.141,,0.144,,0.306,
http://192.168.0.1:8080/paypal/login.php?id=1&t=2,phishing,0.999,,0.999,,1.000,
https://bit.ly/3xYz12,acortador,0.994,,0.994,,1.000,
https://storage.googleapis.com/algo,legítima,0.880,"FP (0.5, FPR≤1%)",0.267,,0.257,
http://paypal-login-seguro.com,phishing,0.981,,0.991,,0.993,
https://www.mercadolibre.com.ar/ofertas,legítima,0.340,,0.376,,0.507,FP (0.5)
https://www.argentina.gob.ar/anses/jubilaciones,legítima,0.238,,0.325,,0.000,
https://github.com/scikit-learn/scikit-learn/issues,legítima,0.203,,0.317,,0.264,


In [31]:
# ¿Qué feature causa cada legítima marcada como FP por los modelos 4c?
for nombre, modelo in entrenados_4c.items():
    print(f"===== {nombre} =====")
    explicar_fp(prueba_4c, f"FP ({nombre.replace('dominio ', '')})", modelo, X_urls_4c, X4c_train, y_train)

===== HGB dominio 4c =====
Ninguna URL legítima de prueba da alta (FP (HGB 4c)).
===== RF dominio 4c =====



https://www.mercadolibre.com.ar/ofertas  (p_phishing = 0.507)


,valor_url,mediana_legítimas,baja_p
feature,,,
tld_ar,1.0,0.0,0.249
cant_puntos,2.0,1.0,0.122
proporcion_digitos,0.0,0.0,0.000
cant_digitos,0.0,0.0,0.000
tiene_punycode,0.0,0.0,0.000


## Conclusiones de la etapa 4c

**En test, la 4c rinde bastante peor que la 4b:**

| | ROC-AUC | recall (0.5) | FPR (0.5) | recall con FPR ≤ 1% | umbral | ms/URL | MB |
|---|---|---|---|---|---|---|---|
| HGB 4b | 0.820 | 0.572 | 3.5% | 0.485 | 0.84 | 2.3 | 0.37 |
| **HGB 4c** | 0.715 | 0.383 | 9.5% | 0.225 | 0.90 | 2.4 | 0.37 |
| RF 4c | 0.700 | 0.392 | 10.8% | 0.221 | 0.92 | 81.5 | 222 |

Nada da sospechosamente alto. Al contrario: gran parte de lo que separaba las clases en la 4b venía de la estructura del hostname completo, es decir, de los subdominios. Con este dataset no se puede saber qué parte de esa señal es real (phishing en `algo-raro.web.app`) y qué parte es artefacto: **casi ninguna legítima del dataset tiene subdominios (2.9%, contra el 54% del phishing)**. Por eso las métricas de test ya no miden bien el uso real. La 4b parece mejor en test justamente porque premia algo que en la navegación real genera falsos positivos.

**En las 13 URLs, la 4c es la primera versión sin falsos positivos (HGB):**
- **Se corrigen googleapis (0.88 → 0.27) y wikipedia (0.63 → 0.26).** Todas las legítimas quedan entre 0.09 y 0.38, lejos del umbral de 0.90.
- **`http://paypal-login.web.app` da 0.998** (marca = 1): la marca en el subdominio se sigue detectando porque las palabras y la marca miran el hostname completo. Los otros 3 phishing y el acortador dan ≥ 0.99.
- RF tiene un falso positivo con umbral 0.5: `mercadolibre.com.ar/ofertas` (0.507), por `tld_ar` (−0.25) y `cant_puntos` (−0.12). Hay pocas `.ar` en el dataset (357 URLs), así que el modelo aprende ruido con esa flag. No llega a su umbral de 0.92.

Ojo: son 13 URLs elegidas a mano. Sirven para detectar problemas groseros, no para medir el rendimiento.

**Qué modelo usar en el backend: HistGradientBoosting.** Frente a RandomForest:
- es unas **35 veces más rápido por URL** (2.4 ms contra 81.5 ms, que en RF se debe al costo de coordinar 300 árboles en paralelo para una sola fila);
- ocupa unas **600 veces menos memoria** (0.37 MB contra 222 MB);
- entrena 10 veces más rápido;
- tiene igual o mejor recall con FPR ≤ 1%.

Sumando la extracción de features (0.05 ms), la respuesta del modelo queda en unos 2.5 ms por URL.

**Versión recomendada: HGB 4c con umbral ~0.90.** Tiene menos recall en el test, pero es la única que no marca subdominios legítimos, que en la navegación real son muy comunes. Con FPR ≤ 1% detecta solo ~22% del phishing del test, así que el ML tiene que funcionar como **una señal más** junto a las reglas y el LLM, no como detector principal. El cuello de botella ahora es el dataset: para recuperar la señal de subdominios sin el atajo hace falta sumar URLs legítimas con subdominios (por ejemplo, de una lista de sitios populares).

# Etapa 6 — Modelo final y función de predicción

Entrenamos HistGradientBoosting (versión 4c) con **todo** el dataset, ya que la evaluación está hecha, y lo guardamos en `models/modelo_phishing.joblib` para que lo use `predict.py`.

In [32]:
# Entrenamiento final sobre todo el dataset (train + test)
datos_dom = pd.read_parquet("data/features_dominio.parquet")
X_final = datos_dom.drop(columns=["URL", "phishing", "dominio"])
y_final = datos_dom["phishing"]
assert list(X_final.columns) == list(X_4c.columns), "Las features no coinciden con la 4c"

inicio = time.perf_counter()
modelo_final = HistGradientBoostingClassifier(random_state=42).fit(X_final, y_final)
print(f"Modelo final entrenado con {len(X_final):,} filas en {time.perf_counter() - inicio:.1f} s")

Modelo final entrenado con 235,795 filas en 1.5 s


In [33]:
# Armado y guardado del artefacto
from datetime import date

import sklearn

UMBRAL_FINAL = 0.90
res_hgb_4c = res_4c["HGB dominio 4c"]
recall_op_4c, umbral_op_4c, _ = recall_con_fpr_max(y_test, res_hgb_4c["proba"])

artefacto = {
    "modelo": modelo_final,
    "features": list(X_final.columns),
    "umbral": UMBRAL_FINAL,
    "medianas_legitimas": {k: float(v) for k, v in X_final[y_final == 0].median().items()},
    "metadatos": {
        "version": "4c",
        "fecha": date.today().isoformat(),
        "entrenado_con": "dataset completo (train + test)",
        "n_filas": len(X_final),
        "sklearn_version": sklearn.__version__,
        "nota_umbral": (
            f"El umbral {UMBRAL_FINAL:.2f} se eligió sobre el test de la 4c "
            f"(FPR <= 1%, umbral exacto {umbral_op_4c:.3f}). El modelo final se reentrenó "
            "con todo el dataset, así que el umbral es aproximado."
        ),
        "metricas_test_4c": {
            "roc_auc": float(res_hgb_4c["roc_auc"]),
            "recall_phishing": float(res_hgb_4c["recall_phishing"]),
            "precision_phishing": float(res_hgb_4c["precision_phishing"]),
            "fpr": float(res_hgb_4c["fpr"]),
            "accuracy": float(res_hgb_4c["accuracy"]),
            "recall_fpr1%": float(recall_op_4c),
            "umbral_fpr1%": float(umbral_op_4c),
        },
    },
}

RUTA_MODELO = Path("models") / "modelo_phishing.joblib"
RUTA_MODELO.parent.mkdir(exist_ok=True)
joblib.dump(artefacto, RUTA_MODELO)

print(f"Guardado en {RUTA_MODELO} ({RUTA_MODELO.stat().st_size / 1e6:.2f} MB)")
print(f"Features ({len(artefacto['features'])}): {artefacto['features']}")
for clave, valor in artefacto["metadatos"].items():
    print(f"{clave}: {valor}")

Guardado en models\modelo_phishing.joblib (0.37 MB)
Features (14): ['longitud_dominio', 'cant_puntos', 'cant_guiones', 'cant_digitos', 'proporcion_digitos', 'usa_ip', 'tiene_punycode', 'entropia_dominio', 'tld_ar', 'tld_comun', 'tld_sospechoso', 'es_acortador', 'cant_palabras_sospechosas', 'marca_fuera_de_dominio']
version: 4c
fecha: 2026-09-24
entrenado_con: dataset completo (train + test)
n_filas: 235795
sklearn_version: 1.9.1
nota_umbral: El umbral 0.90 se eligió sobre el test de la 4c (FPR <= 1%, umbral exacto 0.904). El modelo final se reentrenó con todo el dataset, así que el umbral es aproximado.
metricas_test_4c: {'roc_auc': 0.7154051712153655, 'recall_phishing': 0.3833133413301346, 'precision_phishing': 0.6920115495668913, 'fpr': 0.0950048244637423, 'accuracy': 0.7183924485125858, 'recall_fpr1%': 0.22464347594295614, 'umbral_fpr1%': 0.9039084329083543}


In [34]:
# Prueba de predecir con las 13 URLs
import importlib

import predict

importlib.reload(predict)  # toma el modelo recién guardado


def resumir_features(principales):
    return "; ".join(
        f"{f['feature']} ({f['valor']:g} / {f['mediana_legitima']:g}, +{f['impacto']:.3f})" for f in principales
    ) or "—"


filas = []
for url, esperado in urls_prueba_13.items():
    r = predict.predecir(url)
    filas.append({
        "url": url,
        "esperado": esperado,
        "probabilidad": round(r["probabilidad"], 3),
        "es_sospechoso": r["es_sospechoso"],
        "features que suman sospecha (valor / mediana legítima, impacto)": resumir_features(r["features_principales"]),
    })
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(filas).set_index("url"))

# Chequeo: todas las features devueltas tienen impacto positivo y son como mucho 3
assert all(
    len(r["features_principales"]) <= 3 and all(f["impacto"] > 0 for f in r["features_principales"])
    for r in map(predict.predecir, urls_prueba_13)
)

,esperado,probabilidad,es_sospechoso,"features que suman sospecha (valor / mediana legítima, impacto)"
url,,,,
https://www.bna.com.ar,legítima,0.070,False,—
http://bna-homebanking-verificar.xyz/login,phishing,0.973,True,"cant_guiones (2 / 0, +0.086); cant_palabras_sospechosas (2 / 0, +0.045); tld_comun (0 / 1, +0.006)"
https://www.afip.gob.ar/,legítima,0.124,False,"tld_ar (1 / 0, +0.008)"
http://192.168.0.1:8080/paypal/login.php?id=1&t=2,phishing,0.995,True,"cant_digitos (8 / 0, +0.915); proporcion_digitos (0.727273 / 0, +0.033); cant_puntos (3 / 1, +0.007)"
https://bit.ly/3xYz12,acortador,0.994,True,"es_acortador (1 / 0, +0.657); tld_comun (0 / 1, +0.141); longitud_dominio (6 / 15, +0.046)"
https://storage.googleapis.com/algo,legítima,0.259,False,—
http://paypal-login-seguro.com,phishing,0.993,True,"marca_fuera_de_dominio (1 / 0, +0.026); cant_palabras_sospechosas (2 / 0, +0.020); longitud_dominio (23 / 15, +0.017)"
https://www.mercadolibre.com.ar/ofertas,legítima,0.373,False,"tld_ar (1 / 0, +0.160); cant_puntos (2 / 1, +0.083); entropia_dominio (3.36609 / 3.32782, +0.005)"
https://www.argentina.gob.ar/anses/jubilaciones,legítima,0.282,False,"tld_ar (1 / 0, +0.066); cant_puntos (2 / 1, +0.012)"


In [35]:
# Tiempo por URL (extracción + predicción + explicación), y carga del módulo
PASADAS = 20
inicio = time.perf_counter()
for _ in range(PASADAS):
    for url in urls_prueba_13:
        predict.predecir(url)
ms_url = (time.perf_counter() - inicio) / (PASADAS * len(urls_prueba_13)) * 1000
print(f"predecir: {ms_url:.2f} ms por URL (promedio de {PASADAS * len(urls_prueba_13)} llamadas)")

inicio = time.perf_counter()
importlib.reload(predict)
print(f"Carga del módulo (lee el .joblib): {(time.perf_counter() - inicio) * 1000:.1f} ms, una sola vez al importar")

predecir: 4.56 ms por URL (promedio de 260 llamadas)
Carga del módulo (lee el .joblib): 13.7 ms, una sola vez al importar


In [36]:
# Chequeo: si falta el modelo, importar predict lanza FileNotFoundError con un mensaje claro
RUTA_TEMPORAL = RUTA_MODELO.with_suffix(".joblib.bak")
RUTA_MODELO.rename(RUTA_TEMPORAL)
try:
    importlib.reload(predict)
    print("ERROR: no se lanzó FileNotFoundError")
except FileNotFoundError as e:
    print(f"OK, FileNotFoundError: {e}")
finally:
    RUTA_TEMPORAL.rename(RUTA_MODELO)
    importlib.reload(predict)
    print(f"Modelo restaurado: {RUTA_MODELO.exists()}")

OK, FileNotFoundError: No se encontró el modelo en C:\Users\IPF-2026\Desktop\Hackaton\models\modelo_phishing.joblib. Ejecutá la etapa 6 de main.ipynb para generarlo.
Modelo restaurado: True


## Contrato de `predict.predecir(url)`

```python
from predict import predecir

predecir("http://paypal-login.web.app")
# {
#   "probabilidad": float,          # probabilidad de phishing (0 a 1)
#   "es_sospechoso": bool,          # probabilidad >= umbral
#   "umbral": 0.90,
#   "features_principales": [       # hasta 3, solo las que AUMENTAN la sospecha (puede ser [])
#     {"feature": str, "valor": float, "mediana_legitima": float, "impacto": float},
#   ],
# }
```

- El modelo se carga una sola vez al importar `predict`. Si falta `models/modelo_phishing.joblib`, se lanza `FileNotFoundError`: hay que ejecutar esta etapa para generarlo. `models/` está en `.gitignore`, así que el archivo hay que llevarlo aparte al backend.
- `impacto` = cuánto baja la probabilidad si esa feature toma su valor típico de las legítimas. Es una explicación local y aproximada, pensada para dársela al LLM.
- `predict.py` usa solo `extract_features_dominio` (versión 4c); no importa funciones experimentales.
- Las probabilidades pueden diferir un poco de las de la 4c, porque este modelo se entrenó también con el test. El umbral 0.90 se eligió sobre el test de la 4c, así que es aproximado (queda anotado en `metadatos["nota_umbral"]`).

# Motor — Etapa 1: listas blanca y negra

Demostración de `motor/listas.py`: `es_oficial` compara el dominio registrable contra la lista blanca de marcas argentinas y `esta_en_lista_negra` busca la URL exacta (con y sin query) y el dominio en la lista negra propia y el feed de OpenPhish (si fue descargado).

In [1]:
# Motor — Etapa 1: demostración de listas blanca y negra
import pandas as pd

from motor.listas import es_oficial, esta_en_lista_negra, normalizar_url

urls_demo = [
    "https://www.bna.com.ar/Personas",                          # oficial
    "https://api.mercadopago.com.ar/checkout",                  # oficial, subdominio
    "https://www.afip.gob.ar/landing",                          # oficial (ARCA)
    "https://bna.web.app/login",                                # hosting: no es oficial
    "https://bna.com.ar.evil.com/homebanking",                  # imita el dominio: no es oficial
    "http://bna-homebanking-verificar.xyz/login",               # lista negra por dominio
    "https://reintegros.mercadopago-reintegros.com/cobrar",     # lista negra por dominio (subdominio)
    "https://paypal-login.web.app/verificar?utm_source=sms",    # lista negra por URL exacta (sin query)
]


def describir(resultado):
    return f"{resultado['tipo']}: {resultado['coincidencia']}" if resultado else "—"


pd.DataFrame([
    {
        "url": u,
        "normalizada": normalizar_url(u),
        "marca oficial": es_oficial(u) or "—",
        "lista negra": describir(esta_en_lista_negra(u)),
    }
    for u in urls_demo
]).set_index("url")

,normalizada,marca oficial,lista negra
url,,,
https://www.bna.com.ar/Personas,bna.com.ar/Personas,bna,—
https://api.mercadopago.com.ar/checkout,api.mercadopago.com.ar/checkout,mercadopago,—
https://www.afip.gob.ar/landing,afip.gob.ar/landing,arca,—
https://bna.web.app/login,bna.web.app/login,—,—
https://bna.com.ar.evil.com/homebanking,bna.com.ar.evil.com/homebanking,—,—
http://bna-homebanking-verificar.xyz/login,bna-homebanking-verificar.xyz/login,—,dominio: bna-homebanking-verificar.xyz
https://reintegros.mercadopago-reintegros.com/cobrar,reintegros.mercadopago-reintegros.com/cobrar,—,dominio: mercadopago-reintegros.com
https://paypal-login.web.app/verificar?utm_source=sms,paypal-login.web.app/verificar?utm_source=sms,—,url: paypal-login.web.app/verificar
